### Import libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, ttest_ind
import sys
import os

sys.path.insert(0, '../scripts')
from utils import bootstrap_by_agreement_and_group

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
plt.rcParams.update({'font.size': 14})

from sentence_transformers import SentenceTransformer


### Initialize the model

In [4]:
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", cache_folder="/media/volume/data-backup/language_models")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

### Read the data

In [8]:
fp = "../data/survey/DANTE_Pilot_October 13, 2025_21.48.csv"
df = pd.read_csv(fp)
df = df.drop(index=[0,1])

# remove internal test cases
df = df[df['participantId'] != ""]

# remove unfinished cases
df = df[df['Finished']=="True"]

# remove cases with no summary
df = df.dropna(subset=['summary', 'llm_response_1'])

# remove certain participants (HTTP and DOM errors with erroneous interactions)
error = ["6B5DC61D3F1A4FD59884E09A7390D115", "9C157F56FEEC4F22BAFEEC11763B9759", "F6586134F687435A9064E1C721EBC99F", "AF09ECC56B13406EAA4D174850B78291", "F28F21C7F66946B785FCCA3E4AEB26B2"]
duplicate_ip = ['3E8D695768544BFAAB256B25B286BE36', '2C25201441924CD69CFB850F9E0C25B7', 'ADE33812DFEE4679855302BB37ABFF84']
dnf = ['498B2C7CFDC6421A8DE46C96D56EFBB2', '57A3377A53CD49C3A03640AC86E08698', '6AEA21E0CC19480D95EF78972E41A441']
df = df[df['participantId'].apply(lambda x: x not in error + duplicate_ip + dnf)]

# Convo satisfaction to numeric
str_to_num = {"Strongly disagree": 1, "Somewhat disagree": 2, "Neither agree nor disagree": 3, "Somewhat agree": 4, "Strongly agree": 5}

# Treatment labels to more readable format
df['treatment'] = df['treatment'].apply(lambda x: f"{x.split('_')[0].capitalize()} {x.split('_')[1].capitalize()}")

# Agree vs Disagree
df['agree_disagree'] = "Agree"
df.loc[df['treatment'].apply(lambda x: "Disagree" in x), 'agree_disagree'] = "Disagree"

# Ingroup vs Outgroup
df['ingroup_outgroup'] = "Ingroup"
df.loc[df['treatment'].apply(lambda x: "Outgroup" in x), 'ingroup_outgroup'] = "Outgroup"

In [15]:
def get_all_interactions(df):
    """Combine all user and LLM interactions into a single string for each row"""
    df['all_interactions'] = ""
    for idx, row in df.iterrows():
        interactions = {1:{}, 2:{}, 3:{}, 4:{}, 5:{}}
        interactions[1]['User'] = row['initial_opinion']
        for i in range(1, 6):
            user_col = f'user_response_{i}'
            llm_col = f'llm_response_{i}'
            if pd.notna(row[user_col]):
                interactions[i]['User'] = row[user_col]
            if pd.notna(row[llm_col]):
                interactions[i]['LLM'] = row[llm_col]
        
        df.at[idx, 'all_interactions'] = {k:v for k,v in interactions.items() if v}

In [16]:
get_all_interactions(df)

In [26]:
tmp = df['all_interactions'].iloc[120]

In [27]:
tmp.apply(lambda x: x)

AttributeError: 'dict' object has no attribute 'apply'

In [38]:

queries = [v['User'] for k,v in tmp.items()]
documents = [v['LLM'] for k,v in tmp.items()]
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(queries)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(document_embeddings, document_embeddings)
print(similarity)

tensor([[1.0000, 0.6252, 0.3825],
        [0.6252, 1.0000, 0.4416],
        [0.3825, 0.4416, 1.0000]])


In [35]:
document_embeddings+query_embeddings

array([[-0.10312665,  0.01007168, -0.01113552, ..., -0.0689377 ,
        -0.1294148 , -0.03836921],
       [-0.05026838,  0.02492124, -0.00225873, ..., -0.09800042,
        -0.03451264,  0.00540447],
       [-0.05971353,  0.02231799, -0.00881575, ...,  0.02212181,
        -0.0155665 , -0.03218896]], shape=(3, 1024), dtype=float32)

In [5]:
queries = [
    "What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

# Encode the queries and documents. Note that queries benefit from using a prompt
# Here we use the prompt called "query" stored under `model.prompts`, but you can
# also pass your own prompt via the `prompt` argument
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings, document_embeddings)
print(similarity)


tensor([[0.7646, 0.1414],
        [0.1355, 0.6000]])
